# Eksplorasi data mentah

Jalankan dari atas ke bawah. Notebook ini hanya membaca `data_lake/raw/`, tidak mengubah apa pun.

In [24]:
import json
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_cols(-1)

akar = Path("data_lake/raw")

## 1. Semua kolom mentah

`json_normalize` meratakan JSON bertingkat jadi kolom, misalnya `price.salePrice`.

In [25]:
produk = []
folder = max((akar / "blibli").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    produk.extend(isi["data"]["products"])
blibli_mentah = pl.json_normalize(produk, separator=".", infer_schema_length=None)

produk = []
folder = max((akar / "tokopedia").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    produk.extend(isi[0]["data"]["searchProductV5"]["data"]["products"])
tokped_mentah = pl.json_normalize(produk, separator=".", infer_schema_length=None)

print(blibli_mentah.shape, tokped_mentah.shape)

(1600, 104) (6000, 13)


In [26]:
def kamus(df):
    return pl.DataFrame({
        "kolom": df.columns,
        "tipe": [str(t) for t in df.dtypes],
        "kosong": [df[c].null_count() for c in df.columns],
        "unik": [df[c].n_unique() for c in df.columns],
        "contoh": [str(df[c].drop_nulls().head(1).to_list()) for c in df.columns],
    })

### Kamus kolom Blibli (100 kolom)

`unik` = 1 berarti semua produk bernilai sama. `kosong` tinggi berarti kolom itu hanya ada di sebagian produk.

In [27]:
with pl.Config(tbl_rows=-1):
    display(kamus(blibli_mentah))

kolom,tipe,kosong,unik,contoh
str,str,i64,i64,str
"""id""","""String""",0,1600,"""['HYP-60026-02571']"""
"""sku""","""String""",0,1600,"""['HYP-60026-02571']"""
"""merchantCode""","""String""",0,137,"""['HYP-60026']"""
"""status""","""String""",0,1,"""['AVAILABLE']"""
"""name""","""String""",0,1523,"""['PEPAYA CALIFORNIA 1300 - 1500 GR']"""
"""images""","""List(String)""",0,1598,"""[['https://www.static-src.com/wcsstore/Indraprastha/images/catalog/medium/catalo…"
"""brand""","""String""",0,110,"""['Hypermart']"""
"""itemCount""","""Int64""",0,56,"""[50]"""
"""defaultSku""","""String""",0,1600,"""['HYP-60026-02571-00001-PP-3538904']"""


### Kamus kolom Tokopedia (13 kolom)

In [28]:
with pl.Config(tbl_rows=-1):
    display(kamus(tokped_mentah))

kolom,tipe,kosong,unik,contoh
str,str,i64,i64,str
"""id""","""Int64""",0,5531,"""[100331372217]"""
"""name""","""String""",0,5431,"""['Shura Kurma Mesir Kemasan Ember 700 Gram - Manis Legit Alami']"""
"""url""","""String""",0,5531,"""['https://www.tokopedia.com/surgamatahariid/shura-kurma-mesir-kemasan-ember-700-…"
"""rating""","""String""",0,17,"""['5.0']"""
"""labelGroups""","""List(Struct({'position': String, 'title': String}))""",0,4883,"""[[{'position': 'ri_product_credibility', 'title': '1 terjual'}, {'position': 'ri…"
"""price.number""","""Int64""",0,1573,"""[220000]"""
"""price.original""","""String""",0,447,"""['']"""
"""price.discountPercentage""","""Int64""",0,1,"""[0]"""
"""shop.id""","""Int64""",0,998,"""[7496161994780085088]"""


### Satu produk, semua field (ganti angka baris untuk melihat produk lain)

In [29]:
for kolom, nilai in blibli_mentah.row(0, named=True).items():
    print(f"{kolom:35} {nilai}")

id                                  HYP-60026-02571
sku                                 HYP-60026-02571
merchantCode                        HYP-60026
status                              AVAILABLE
name                                PEPAYA CALIFORNIA 1300 - 1500 GR
images                              ['https://www.static-src.com/wcsstore/Indraprastha/images/catalog/medium/catalog-image/MTA-10881321/hypermart_pepaya_california_1400_gr_-_buah_full02_vs7jzf78.jpeg']
brand                               Hypermart
itemCount                           50
defaultSku                          HYP-60026-02571-00001-PP-3538904
itemSku                             HYP-60026-02571-00001
tags                                ['REGULAR', 'XXX_BOUGHT_IN_LAST_MONTH', 'SHIPPING_ETD', 'BEST_SELLING', 'FREE_SHIPPING']
formattedId                         ps--HYP-60026-02571
url                                 /p/pepaya-california-1300-1500-gr/ps--HYP-60026-02571?ds=HYP-60026-02571-00001&source=CATEGORY&sid=71fc6

In [30]:
for kolom, nilai in tokped_mentah.row(0, named=True).items():
    print(f"{kolom:35} {nilai}")

id                                  100331372217
name                                Shura Kurma Mesir Kemasan Ember 700 Gram - Manis Legit Alami
url                                 https://www.tokopedia.com/surgamatahariid/shura-kurma-mesir-kemasan-ember-700-gram-manis-legit-alami-1731377847620896608?extParam=ivf%3Dtrue%26search_id%3D2026110702334958238E9AE2DD30220M3B
rating                              5.0
labelGroups                         [{'position': 'ri_product_credibility', 'title': '1 terjual'}, {'position': 'ri_product_offer', 'title': 'Hemat s.d  8% Pakai Bonus'}, {'position': 'final_price', 'title': 'Rp220.000'}, {'position': 'ri_product_offer_v2', 'title': 'Hemat s.d  8% Pakai Bonus'}, {'position': 'ri_product_offer_v2', 'title': 'Bisa COD'}]
price.number                        220000
price.original                      
price.discountPercentage            0
shop.id                             7496161994780085088
shop.name                           surgamatahari.id
shop.c

### Satu produk Blibli berdasarkan ID (bandingkan dengan halaman produknya)

ID ada di URL produk: `blibli.com/p/.../ps--NAS-70965-00407`. Data ini dari tanggal terakhir, jadi bisa beda dengan halaman hari ini.

In [31]:
cari_id = "NAS-70965-00407"
for kolom, nilai in blibli_mentah.filter(pl.col("id") == cari_id).row(0, named=True).items():
    print(f"{kolom:42} {nilai}")

id                                         NAS-70965-00407
sku                                        NAS-70965-00407
merchantCode                               NAS-70965
status                                     AVAILABLE
name                                       Apel Fuji Super Fresh 1 Kg
images                                     ['https://www.static-src.com/wcsstore/Indraprastha/images/catalog/medium/catalog-image/MTA-164386682/no_brand_apel_fuji_super_fresh_1_kg_full02_896bc57e.jpg', 'https://www.static-src.com/wcsstore/Indraprastha/images/catalog/medium/catalog-image/MTA-164386682/no_brands_apel_fuji_fresh_1_kg_full01_5da35671.jpg']
brand                                      no brand
itemCount                                  2
defaultSku                                 NAS-70965-00407-00001-PP-3530980
itemSku                                    NAS-70965-00407-00001
tags                                       ['REGULAR', 'XTRA_PROMO', 'FREE_SHIPPING']
formattedId                

## 2. Tabel ringkas (11 kolom pilihan)

In [32]:
baris = []
folder = max((akar / "blibli").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    for p in isi["data"]["products"]:
        baris.append({
            "id": p["id"],
            "nama": p["name"],
            "harga": p["price"]["salePrice"],
            "harga_awal": p["price"]["listPrice"],
            "toko": p["merchantName"],
            "brand": p["brand"],
            "kota": p["location"],
            "kategori": p["categoryNameHierarchy"][-1],
            "terjual": p.get("soldCountTotal"),
            "rating": p["review"]["absoluteRating"],
            "berkas": berkas.name,
        })
blibli = pl.DataFrame(baris)
print(folder.name, blibli.shape)

date=2026-11-07 (1600, 11)


In [33]:
baris = []
folder = max((akar / "tokopedia").glob("date=*"))
for berkas in sorted(folder.glob("*.json")):
    isi = json.loads(berkas.read_text(encoding="utf-8"))
    for p in isi[0]["data"]["searchProductV5"]["data"]["products"]:
        label = {g["position"]: g["title"] for g in p["labelGroups"]}
        baris.append({
            "id": str(p["id"]),
            "nama": p["name"],
            "harga": p["price"]["number"],
            "harga_awal": p["price"]["original"],
            "toko": p["shop"]["name"],
            "tier": p["shop"]["tier"],
            "kota": p["shop"]["city"],
            "kategori": p["category"]["breadcrumb"],
            "terjual": label.get("ri_product_credibility"),
            "rating": p["rating"],
            "berkas": berkas.name,
        })
tokped = pl.DataFrame(baris)
print(folder.name, tokped.shape)

date=2026-11-07 (6000, 11)


### Lihat sekilas

In [34]:
display(blibli.head(20))
display(tokped.head(20))

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""HYP-60026-02571""","""PEPAYA CALIFORNIA 1300 - 1500 GR""",22350,22350,"""Hypermart Flagship Store""","""Hypermart""","""Kab. Gresik""","""Buah Lokal""",6952,4.7,"""BU-1000190_page_001.json"""
"""RAM-70107-10697""","""DRAGON FRUIT RED""",38400,38400,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",1544,4.8,"""BU-1000190_page_001.json"""
"""HYP-60026-23815""","""JERUK CARACARA 540 - 600 GR""",24300,24300,"""Hypermart Flagship Store""","""Hypermart""","""Kab. Gresik""","""Buah Lokal""",12,5.0,"""BU-1000190_page_001.json"""
"""RAM-70107-29314""","""MANGGA MANALAGI 170 - 180 G""",6800,6800,"""Ranch Market Flagship Store""","""no brand""","""Kota Surabaya""","""Buah Lokal""",157,4.2,"""BU-1000190_page_001.json"""
"""HYP-60026-23829""","""JERUK CLEMENGOLD 540 - 600 GR""",53940,53940,"""Hypermart Flagship Store""","""Hypermart""","""Kab. Gresik""","""Buah Lokal""",11,1.0,"""BU-1000190_page_001.json"""
"""HYP-60026-14414""","""MANGGA BUDIRAJA 880-890 GR""",22160,23050,"""Hypermart Flagship Store""","""FRUIT""","""Kab. Gresik""","""Buah Lokal""",44,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23761""","""ALPUKAT RAJA 1125 - 1250 GR""",72370,72370,"""Hypermart Flagship Store""","""Hypermart""","""Kab. Bangkalan""","""Buah Lokal""",2,0.0,"""BU-1000190_page_001.json"""
"""RAM-70107-17478""","""MANGGA BUDIRAJA 470 - 500 G""",26500,26500,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",71,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23787""","""SEMANGKA KUNING 4950 - 5500 GR""",123750,123750,"""Hypermart Flagship Store""","""Hypermart""","""Kab. Bangkalan""","""Buah Lokal""",1,0.0,"""BU-1000190_page_001.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""100331372217""","""Shura Kurma Mesir Kemasan Ember 700 Gram - Manis Legit Alami""",220000,"""""","""surgamatahari.id""",2,"""Depok""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_001.json"""
"""102071771069""","""Avocadron Alpukat Mentega Mini [Paket 2 Kg]""",105600,"""Rp136.730""","""AVOCADRON""",2,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""100+ terjual""","""5.0""","""page_001.json"""
"""10705788896""","""Anggur Red Globe Premium Import Untuk Add On dan Beli dengan Gramasi""",54450,"""Rp55.000""","""sevenrosecoid""",2,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""80+ terjual""","""5.0""","""page_001.json"""
"""13835356583""","""TAMBAH BUAH DAN SAYUR Jeruk Peras Fresh 1 kg - Buah Sayur Segar & Halal - Packin…",28000,"""""","""Tambah Buah Sayur""",2,"""Jakarta Timur""","""makanan-minuman/buah/buah-buahan""","""30+ terjual""","""5.0""","""page_001.json"""
"""2652345561""","""Jeruk Ponkam - Sayur Kendal""",126000,"""""","""Sayur Kendal Menteng""",2,"""Kab. Sleman""","""makanan-minuman/buah/buah-buahan""","""23 terjual""","""5.0""","""page_001.json"""
"""10529450258""","""PEAR MADU PREMIUM AVOCADRON NEXT DAY""",65000,"""""","""AVOCADRON""",2,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""5.0""","""page_001.json"""
"""869816887""","""Pir Celina I Pear Forelle Afrika (1 KG)""",69000,"""""","""RAJA ALPUKAT""",2,"""Jakarta Timur""","""makanan-minuman/buah/buah-buahan""","""80+ terjual""","""5.0""","""page_001.json"""
"""2315850352""","""Castle Farms Kurma Ajwa 500gr Natural Organik | Kurma Nabi Original Madinah | Ku…",143000,"""Rp173.650""","""Castle Farms Indonesia""",2,"""Bekasi""","""makanan-minuman/buah/buah-buahan""","""1rb+ terjual""","""5.0""","""page_001.json"""
"""103715086232""","""Buah Delima merah Import Rasa Manis Asam Segar by Tambah Buah Sayur""",55000,"""""","""Tambah Buah Sayur""",2,"""Jakarta Timur""","""makanan-minuman/buah/buah-buahan""","""50+ terjual""","""5.0""","""page_001.json"""


### Nilai kosong per kolom

In [35]:
display(blibli.null_count())
display(tokped.null_count())

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,506,0,0


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,40,0,0


### Baris acak (bukan hanya halaman 1)

In [36]:
display(blibli.sample(20, seed=1))
display(tokped.sample(20, seed=1))

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""SA7-70003-00427""","""Jeruk Sunkist RRC - 1kg""",45000,45000,"""SAYURKITA2017""","""sayur kita""","""Kota Jakarta Pusat""","""Buah Impor""",19,0.0,"""BU-1000191_page_013.json"""
"""ANB-70067-00214""","""Buah Pear sweet/pear madu fresh Import 1 dus""",335000,335000,"""Kenken juice""","""no brand""","""Kota Jakarta Timur""","""Buah Impor""",null,0.0,"""BU-1000191_page_010.json"""
"""SES-70872-01432""","""Pisang Barangan (450 - 550 gram / pack)""",17200,17200,"""Segari Supermarket Flagship Store""","""no brand""","""Kota Bekasi""","""Buah Lokal""",28,5.0,"""BU-1000190_page_004.json"""
"""HAM-70126-00399""","""HarvestMoon Jeruk Shantang per krat""",365000,365000,"""Harvest Moon Jakarta""","""no brand""","""Kota Jakarta Barat""","""Buah Impor""",2,0.0,"""BU-1000191_page_010.json"""
"""PAB-70112-00803""","""Markisa manis 1kg""",52934,53200,"""Pasar badung""","""buah & mayur""","""Kota Denpasar""","""Buah Lokal""",null,0.0,"""BU-1000190_page_008.json"""
"""RAM-70107-03329""","""PUMPKIN MINI""",39900,39900,"""Ranch Market Flagship Store""","""Pumpkin""","""Kota Malang""","""Buah Impor""",null,0.0,"""BU-1000191_page_004.json"""
"""IBF-60022-00202""","""Buah Sirsat segar - Harga per butir buah sirsat""",17000,17000,"""IBAZA fruitsBOX""","""IBAZA fruitsBOX""","""Kota Semarang""","""Buah Impor""",1,0.0,"""BU-1000191_page_020.json"""
"""RAM-70107-09635""","""APEL RED DELICIOUS BESAR 470 - 500 G""",43800,43800,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Impor""",547,4.8,"""BU-1000191_page_001.json"""
"""RAM-70107-10667""","""MANGGA HARUMANIS PREMIUM 470 - 500 G""",13800,13800,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Jakarta Pusat""","""Buah Lokal""",343,4.9,"""BU-1000190_page_004.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""1666143359""","""Original Kurma Medjool Palestine Premium 1kg - Medjool Jumbo""",350000,"""""","""Elshanum Pusat""",3,"""Jakarta Pusat""","""makanan-minuman/buah/buah-buahan""","""12 terjual""","""4.3""","""page_081.json"""
"""1696522945""","""KURMA MEDJOOL MEDJOL PREMIUM ISTIMEWAH KURMA NABI QURMA ASLI ARAB""",210320,"""Rp239.000""","""Marf Store""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""4.7""","""page_075.json"""
"""100767638499""","""Buah salak gula pasir manis garing dan fresh 1 kg""",84480,"""Rp88.000""","""99Fruit Shop""",3,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""21 terjual""","""5.0""","""page_010.json"""
"""10367715663""","""SEMANGKA MERAH TANPA BIJI (3,5 - 4 KG)""",89500,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""40+ terjual""","""4.7""","""page_075.json"""
"""9436480209""","""Jeruk bali madu kupas/pamelo kupas segar/pack""",55800,"""""","""99Fruit Shop""",3,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""6 terjual""","""5.0""","""page_019.json"""
"""281180536""","""Jeruk Kasturi Kalamansi Son Kit Seng Kit Kit Kia Besar khas Pontianak Kalimantan…",10236,"""""","""Sandyptk""",3,"""Pontianak""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""4.9""","""page_059.json"""
"""10413390426""","""Anggur Shine Muscat Buah Import Buah Segar Seedless Pilihan""",52440,"""Rp76.000""","""RAYA BUAH SEGAR""",1,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""70+ terjual""","""5.0""","""page_099.json"""
"""2627834370""","""BUAH BIT/ BEET ROOT PER Kg""",21000,"""""","""Toko Prima Buah & Sayur""",3,"""Tangerang""","""makanan-minuman/buah/buah-buahan""","""500+ terjual""","""4.9""","""page_053.json"""
"""1607020871""","""umbi talas pulen 1kg""",10000,"""""","""fitriherball92""",3,"""Kab. Bandung Barat""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_010.json"""


### Termurah dan termahal

In [37]:
display(blibli.sort("harga").head(15))
display(blibli.sort("harga", descending=True).head(15))
display(tokped.sort("harga").head(15))
display(tokped.sort("harga", descending=True).head(15))

id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""NAS-70965-00133""","""Asam Jawa Matang 1 pack""",1600,1600,"""Namora Sayur""","""no brand""","""Kota Jakarta Timur""","""Buah Lokal""",4,0.0,"""BU-1000190_page_010.json"""
"""RUS-70010-00140""","""Rumah Sayur Express Mangga Muda [250 gram]""",4875,4875,"""Rumah Sayur Express""","""Rumah Sayur""","""Kota Jakarta Selatan""","""Buah Lokal""",62,4.7,"""BU-1000190_page_007.json"""
"""WI8-70008-00230""","""buah lemon eureka sunpride original indonesia""",5000,5000,"""wisata88""","""Sunpride""","""Kota Tangerang Selatan""","""Buah Lokal""",null,0.0,"""BU-1000190_page_009.json"""
"""NAS-70965-00256""","""Jeruk Limau 1 Pack""",5000,5000,"""Namora Sayur""","""no brand""","""Kota Jakarta Timur""","""Buah Lokal""",1,3.0,"""BU-1000190_page_010.json"""
"""SUG-60058-00951""","""Jeruk Nipis 200g""",5000,5000,"""Super Grosir Mama Flagship Store""","""no brand""","""Kota Jakarta Selatan""","""Buah Lokal""",null,0.0,"""BU-1000190_page_010.json"""
"""WAS-70184-00031""","""Jeruk Limo (2 Buah)""",5000,5000,"""Warung Sayur Sebelah""","""no brand""","""Kota Tangerang""","""Frozen Food""",12,2.0,"""BU-1000190_page_012.json"""
"""PAS-70072-00347""","""Jagung Manis 1 buah""",6000,6000,"""Pak Sayur 10""","""Paksayur""","""Kota Jakarta Pusat""","""Buah Lokal""",254,4.8,"""BU-1000190_page_012.json"""
"""WAR-70076-00259""","""buah asam""",6000,6000,"""Warung Rangkuti""","""no brand""","""Kota Depok""","""Sayur""",1,0.0,"""BU-1000190_page_012.json"""
"""CEF-70023-00536""","""Pisang Tanduk Fresh - Pisang Agung | 1 Buah [ Harga Per PCS ]""",6000,6000,"""Centre Foods Jakarta""","""CFI Fresh""","""Kota Jakarta Barat""","""Buah Lokal""",null,0.0,"""BU-1000190_page_012.json"""


id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""SAR-70214-00807""","""DRIED BLUEBERRY - 1 DUS (10 KG) | Blueberry Kering 1 DUS 10 KG Premium Healthy U…",3200000,3200000,"""Favore""","""no brand""","""Kota Medan""","""Buah Impor""",null,0.0,"""BU-1000191_page_006.json"""
"""RAM-70107-32769""","""MELON CROWN JEPANG""",2350000,2350000,"""Ranch Market Flagship Store""","""no brand""","""Kota Jakarta Selatan""","""Buah Impor""",null,0.0,"""BU-1000191_page_018.json"""
"""GBE-70000-00120""","""SU Brand Manisan Buah Kundur 250g - Medium Pack - 50Pcs""",1253105,1253105,"""GBM Express""","""SU BRAND""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_007.json"""
"""CHB-70036-00061""","""BUAH APEL ENVY FRESH IMPORT PER DUS""",950000,950000,"""Chandra Buah""","""buah & mayur""","""Kota Jakarta Barat""","""Buah Impor""",3,5.0,"""BU-1000191_page_017.json"""
"""TOM-71006-02987""","""Sunsweet Buah Prunes 200 gr×12 Pcs""",942000,942000,"""Toko MAXD""","""Sunsweet""","""Kota Medan""","""Buah Impor""",null,0.0,"""BU-1000191_page_014.json"""
"""CHB-70036-00079""","""Buah Anggur Hijau Tanpa Biji 1dus""",900000,900000,"""Chandra Buah""","""buah & mayur""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_010.json"""
"""HAM-70126-00044""","""HarvestMoon Apel Hijau Per Dus""",890000,925000,"""Harvest Moon Jakarta""","""no brand""","""Kota Jakarta Barat""","""Minyak Goreng Halal""",null,0.0,"""BU-1000191_page_016.json"""
"""CHB-70036-00044""","""Buah Anggur Hitam Manis Lonjong Moondrop 1dus 8kg-/+""",875000,875000,"""Chandra Buah""","""buah & mayur""","""Kota Jakarta Barat""","""Buah Impor""",null,0.0,"""BU-1000191_page_005.json"""
"""HAL-70041-00092""","""Fruit Order""",874000,874000,"""Harvest Line""","""no brand""","""Kab. Tangerang""","""Buah Impor""",1,5.0,"""BU-1000191_page_016.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""2371668452""","""jeruk limo harga per biji.""",1000,"""""","""Outlet Toni Sayur""",1,"""Bogor""","""makanan-minuman/buah/buah-buahan""","""23 terjual""","""5.0""","""page_095.json"""
"""811731989""","""JERUK NIPIS SEGAR | tukang sayur jkt""",1800,"""""","""tukangsayurjkt""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""1rb+ terjual""","""4.9""","""page_058.json"""
"""1041151326""","""KIRIM INSTAN Jeruk nipis 1 buah""",2000,"""""","""SayurExpress.id""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""500+ terjual""","""5.0""","""page_022.json"""
"""2362417583""","""JERUK NIPIS PER BUAH""",2500,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""5.0""","""page_006.json"""
"""2362417583""","""JERUK NIPIS PER BUAH""",2500,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""5.0""","""page_007.json"""
"""896623559""","""Jeruk Limo""",2500,"""""","""waroeng sayur""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""750+ terjual""","""5.0""","""page_031.json"""
"""12303756062""","""JERUK NIPIS SEGAR (250 GRAM)""",2500,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""500+ terjual""","""4.9""","""page_052.json"""
"""406051633""","""Jeruk Peras - Bakoel Sayur Online""",2800,"""""","""Bakoel Sayur Online""",3,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""10rb+ terjual""","""4.8""","""page_066.json"""
"""12303761673""","""LEMON LOKAL SEGAR (500 GRAM)""",2900,"""""","""tukatukushop.id""",3,"""Tangerang Selatan""","""makanan-minuman/buah/buah-buahan""","""250+ terjual""","""4.8""","""page_065.json"""


id,nama,harga,harga_awal,toko,tier,kota,kategori,terjual,rating,berkas
str,str,i64,str,str,i64,str,str,str,str,str
"""15901621686""","""bibit kelapa kopyor custom harga grosir 150 kelapa genjah dari pati""",6500000,"""""","""Bismillah Pete Kopyor PATI""",3,"""Kab. Pati""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_020.json"""
"""7472067550""","""persimon korea original kesemek import korea - 1kg""",4320000,"""""","""Jakfresh MARKET""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""4 terjual""","""5.0""","""page_033.json"""
"""6981647163""","""buah anggur nagano asli jepang / anggur import jepang original 700gr""",2815000,"""""","""ASSISTANT-05 shop""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_045.json"""
"""1651715858""","""Original Kurma Medjool Natural Delights Premium 5kg - Medjool Jumbo""",2750000,"""""","""Elshanum Pusat""",3,"""Jakarta Pusat""","""makanan-minuman/buah/buah-buahan""","""15 terjual""","""5.0""","""page_037.json"""
"""12103371725""","""BUAH CHERRY RED TASMANIA PREMIUM FRESH AUSTRALIA 43° SOUTH BOX""",2690000,"""""","""Fresh Fruit Shop1""",1,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""12 terjual""","""5.0""","""page_091.json"""
"""12103371725""","""BUAH CHERRY RED TASMANIA PREMIUM FRESH AUSTRALIA 43° SOUTH BOX""",2690000,"""""","""Fresh Fruit Shop1""",1,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""12 terjual""","""5.0""","""page_092.json"""
"""9631030506""","""Cherry Merah | buah ceri red Usa 5 kg Manis segar/ Box""",2399000,"""""","""PartnerBuah""",3,"""Jakarta Barat""","""makanan-minuman/buah/buah-buahan""","""2 terjual""","""5.0""","""page_018.json"""
"""3619342666""","""Buah naga kuning super import Manis 2kg""",1970000,"""""","""ASSISTANT-05 shop""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""4 terjual""","""5.0""","""page_038.json"""
"""6626711436""","""Delima China Segar Pomegranate RRC 1 Dus dengan Kualitas Terbaik dan Refraktomet…",1940000,"""""","""ASSISTANT-05 shop""",3,"""Jakarta Selatan""","""makanan-minuman/buah/buah-buahan""","""1 terjual""","""5.0""","""page_041.json"""


### Cari kata di nama produk (ganti `kata` lalu jalankan ulang)

Coba: `kurma`, `potong`, `kering`, `manisan`, `sisir`, `pack`.

In [38]:
kata1="mangga"
display(tokped_mentah.filter(pl.col("name").str.contains(f"(?i){kata1}")))

id,name,url,rating,labelGroups,price.number,price.original,price.discountPercentage,shop.id,shop.name,shop.city,shop.tier,category.breadcrumb
i64,str,str,str,list[struct[2]],i64,str,i64,i64,str,str,i64,str
13843891847,"""TAMBAH BUAH DAN SAYUR MANGGA HARUM MANIS FROZEN 500 GR HALAL 100% FRESH Tambah B…","""https://www.tokopedia.com/tambahbuahsayur/tambah-buah-dan-sayur-mangga-harum-man…","""5.0""","[{""ri_product_credibility"",""30+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",29999,"""""",0,8933388,"""Tambah Buah Sayur""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
7241408517,"""BUAH POTONG FRESH, MANGGA-ALPUKAT-MELON-PEPAYA-BUAH IMPOR-JAMBU KRISTAL-APEL-SEM…","""https://www.tokopedia.com/rajaalpukat/buah-potong-fresh-mangga-alpukat-melon-pep…","""5.0""","[{""ri_product_credibility"",""750+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",20000,"""""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
7286804348,"""[PAKET 5 KG] MANGGA HARUM MANIS AVOCADRON""","""https://www.tokopedia.com/avocadron/paket-5-kg-mangga-harum-manis-avocadron?extP…","""5.0""","[{""ri_product_credibility"",""100+ terjual""}, {""nett_price"",""Rp256.500""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",256500,"""Rp285.000""",0,1945794,"""AVOCADRON""","""Jakarta Barat""",2,"""makanan-minuman/buah/buah-buahan"""
100652285365,"""Avocadron Mangga Harum Manis [Paket 2 Kg]""","""https://www.tokopedia.com/avocadron/avocadron-mangga-harum-manis-paket-2-kg-1731…","""5.0""","[{""ri_product_credibility"",""40+ terjual""}, {""ri_product_offer"",""Hemat s.d 8% Pakai Bonus""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",114000,"""""",0,1945794,"""AVOCADRON""","""Jakarta Barat""",2,"""makanan-minuman/buah/buah-buahan"""
1002768127,"""HARGA GROSIR MANGGA ALPUKAT SATU BOX (10 KG)""","""https://www.tokopedia.com/rajaalpukat/harga-grosir-mangga-alpukat-satu-box-10-kg…","""5.0""","[{""ri_product_credibility"",""40+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",430000,"""""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
2362761747,"""Mangga Matang Pohon - Sayur Kendal""","""https://www.tokopedia.com/sayurkendal/mangga-matang-pohon-sayur-kendal?extParam=…","""5.0""","[{""ri_product_credibility"",""70+ terjual""}, {""ri_product_offer"",""Hemat s.d 8% Pakai Bonus""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",77000,"""""",0,8386110,"""Sayur Kendal Menteng""","""Kab. Sleman""",2,"""makanan-minuman/buah/buah-buahan"""
100625997762,"""Buah Mangga Harum Manis Tua Pohon-Raja Alpukat""","""https://www.tokopedia.com/rajaalpukat/buah-mangga-harum-manis-tua-pohon-raja-alp…","""5.0""","[{""ri_product_credibility"",""500+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",35000,"""""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
102850029426,"""Mangga Apel Rujak Berat 500 Gr dan 1 Kg By Tambah Buah Sayur""","""https://www.tokopedia.com/tambahbuahsayur/tambah-buah-sayur-mangga-muda-fresh-je…","""5.0""","[{""ri_product_credibility"",""60+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Bisa COD""}]",30000,"""""",0,8933388,"""Tambah Buah Sayur""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""
1859479644,"""Grosir Mangga Harum Manis 1 Box (10 Kg)""","""https://www.tokopedia.com/rajaalpukat/grosir-mangga-harum-manis-1-box-10-kg?extP…","""5.0""","[{""ri_product_credibility"",""60+ terjual""}, {""overlay_3"",""Beli Lokal""}, … {""ri_product_offer_v2"",""Hemat s.d 8% Pakai Bonus""}]",307200,"""Rp320.000""",0,2447242,"""RAJA ALPUKAT""","""Jakarta Timur""",2,"""makanan-minuman/buah/buah-buahan"""


In [39]:
kata = "mangga"
display(blibli.filter(pl.col("nama").str.contains(f"(?i){kata}")))


id,nama,harga,harga_awal,toko,brand,kota,kategori,terjual,rating,berkas
str,str,i64,i64,str,str,str,str,i64,f64,str
"""RAM-70107-29314""","""MANGGA MANALAGI 170 - 180 G""",6800,6800,"""Ranch Market Flagship Store""","""no brand""","""Kota Surabaya""","""Buah Lokal""",157,4.2,"""BU-1000190_page_001.json"""
"""HYP-60026-14414""","""MANGGA BUDIRAJA 880-890 GR""",22160,23050,"""Hypermart Flagship Store""","""FRUIT""","""Kab. Gresik""","""Buah Lokal""",44,5.0,"""BU-1000190_page_001.json"""
"""RAM-70107-17478""","""MANGGA BUDIRAJA 470 - 500 G""",26500,26500,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",71,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-11159""","""MANGGA HARUM MANIS 550-580 GR""",9220,14440,"""Hypermart Flagship Store""","""no brand""","""Kab. Gresik""","""Buah Lokal""",4014,4.6,"""BU-1000190_page_001.json"""
"""RAM-70107-19157""","""MANGGA HARUMANIS LILY 470 - 500 G""",21300,21300,"""Ranch Market Flagship Store""","""Ranch Market""","""Kota Surabaya""","""Buah Lokal""",894,4.6,"""BU-1000190_page_001.json"""
"""RAM-70107-28584""","""MANGGA GEDONG GINCU""",16500,16500,"""Ranch Market Flagship Store""","""Farmers Market""","""Kota Surabaya""","""Buah Lokal""",103,4.5,"""BU-1000190_page_001.json"""
"""RAM-70107-04265""","""MANGGA HARUMANIS SULUHAN""",39900,49900,"""Ranch Market Flagship Store""","""MANGGA""","""Kota Surabaya""","""Buah Lokal""",18,5.0,"""BU-1000190_page_001.json"""
"""HYP-60026-23863""","""MANGGA INDRAMAYU 720 - 800 GR""",15920,19920,"""Hypermart Flagship Store""","""Hypermart""","""Kota Surabaya""","""Buah Lokal""",73,4.6,"""BU-1000190_page_002.json"""
"""HYP-60026-14382""","""MANGGA HARUMANIS SUPER 500 -550GR""",9850,14240,"""Hypermart Flagship Store""","""FRUIT""","""Kab. Gresik""","""Buah Lokal""",1203,4.5,"""BU-1000190_page_003.json"""


### Frekuensi nilai

In [40]:
display(blibli["kategori"].value_counts(sort=True))
display(tokped["kategori"].value_counts(sort=True))
display(tokped["terjual"].value_counts(sort=True))

kategori,count
str,u32
"""Buah Lokal""",767
"""Buah Impor""",759
"""Buah""",38
"""Sayur""",18
"""Fresh Produk""",12
"""Makanan & Minuman Halal""",1
"""Frozen Food""",1
"""Bumbu Rempah""",1
"""Biskuit & Snack Halal""",1


kategori,count
str,u32
"""makanan-minuman/buah/buah-buahan""",6000


terjual,count
str,u32
"""100+ terjual""",750
"""250+ terjual""",450
"""30+ terjual""",305
"""2 terjual""",255
"""1 terjual""",235
"""500+ terjual""",234
"""1rb+ terjual""",232
"""40+ terjual""",216
"""3 terjual""",210


## 3. Satuan setelah angka

Kata pertama setelah angka di nama produk: `500 gr`, `1 kg`, `2 pcs`, dan seterusnya.

In [41]:
gabung = pl.concat([
    blibli.select(pl.lit("blibli").alias("sumber"), "id", "nama", "harga"),
    tokped.select(pl.lit("tokopedia").alias("sumber"), "id", "nama", "harga"),
])
satuan = gabung.select(pl.col("nama").str.extract(r"(?i)\d\s*([a-z]+)\b", 1).alias("satuan"))
with pl.Config(tbl_rows=-1):
    display(satuan["satuan"].value_counts(sort=True))

satuan,count
str,u32
null,1982
"""kg""",1506
"""gr""",859
"""Kg""",519
"""gram""",504
"""KG""",405
"""GR""",185
"""Gram""",181
"""g""",172


### Contoh nama per satuan (ganti `cari` lalu jalankan ulang)

Huruf besar/kecil diabaikan: `kg`, `Kg`, dan `KG` dihitung sama. Coba: `sisir`, `x`, `s`, `per`, `ml`, `pack`.

In [42]:
cari = "sisir"
dengan_satuan = gabung.with_columns(
    pl.col("nama").str.extract(r"(?i)\d\s*([a-z]+)\b", 1).str.to_lowercase().alias("satuan")
)
with pl.Config(tbl_rows=-1):
    display(dengan_satuan["satuan"].value_counts(sort=True))
    display(dengan_satuan.filter(pl.col("satuan") == cari))

satuan,count
str,u32
"""kg""",2430
null,1982
"""gr""",1115
"""gram""",796
"""g""",261
"""pcs""",209
"""buah""",206
"""grm""",103
"""sisir""",83


sumber,id,nama,harga,satuan
str,str,str,i64,str
"""blibli""","""BAF-70007-00022""","""Pisang Kepok 1 Sisir""",43394,"""sisir"""
"""blibli""","""BAF-70007-00020""","""Pisang Tanduk (Pisang Candi) 1 Sisir""",35000,"""sisir"""
"""blibli""","""BAF-70007-00021""","""Pisang Barlin 1 Sisir""",17500,"""sisir"""
"""blibli""","""BAF-70007-00029""","""Pisang Mas 1 Sisir""",36443,"""sisir"""
"""blibli""","""BAF-70007-00024""","""Pisang Raja 1 Sisir""",44549,"""sisir"""
"""blibli""","""BAF-70007-00025""","""Pisang Susu 1 Sisir""",38048,"""sisir"""
"""blibli""","""BAF-70007-00027""","""Buah Pisang Mas / Emas 1 Sisir""",36000,"""sisir"""
"""blibli""","""BAF-70007-00023""","""Pisang Ambon 1 Sisir""",42750,"""sisir"""
"""blibli""","""FRX-60021-00853""","""Pisang Uli 1 sisir FreshBox""",28500,"""sisir"""


## 4. Berat cadangan: `variant` Blibli dan URL Tokopedia

Semua tanggal digabung, satu baris per produk. `dari_nama` / `dari_variant` / `dari_url` = teks berat pertama yang ketemu (belum diubah ke gram).

URL Tokopedia dibuat dari nama **asli** produk, dan semua tanda baca jadi `-`. Jadi `1/2 kg`, `1.2 kg`, dan `1 - 2 kg` sama-sama jadi `1 2 kg`; baris seperti itu ditandai `ambigu`.

In [43]:
BERAT = r"(?i)(\d+(?:[.,/]\d+)?\s*(?:kg|kilogram|kilo|gram|gr|grm|g|ons)\b)"
AMBIGU = r"(?i)\b\d+ \d+ ?(?:kg|kilogram|kilo|gram|gr|grm|g|ons)\b"

baris = []
for berkas in (akar / "blibli").glob("date=*/*.json"):
    for p in json.loads(berkas.read_text(encoding="utf-8"))["data"]["products"]:
        baris.append({"id": p["id"], "nama": p["name"], "variant": p.get("variant")})
blibli_semua = pl.DataFrame(baris).unique("id").with_columns(
    dari_nama=pl.col("nama").str.extract(BERAT, 1),
    dari_variant=pl.col("variant").str.extract(BERAT, 1),
)

baris = []
for berkas in (akar / "tokopedia").glob("date=*/*.json"):
    for p in json.loads(berkas.read_text(encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]:
        baris.append({"id": str(p["id"]), "nama": p["name"], "url": p["url"]})
tokped_semua = (
    pl.DataFrame(baris).unique("id")
    # Ambil bagian setelah nama toko, buang ID angka panjang di ujung, ganti "-" jadi spasi.
    .with_columns(slug=pl.col("url").str.extract(r"tokopedia\.com/[^/]+/([^?]+)", 1)
                  .str.replace(r"(-\d{15,})+$", "").str.replace_all("-", " "))
    .with_columns(
        dari_nama=pl.col("nama").str.extract(BERAT, 1),
        dari_url=pl.col("slug").str.extract(BERAT, 1),
        ambigu=pl.col("slug").str.contains(AMBIGU),
    )
)

print("Blibli", blibli_semua.height, "produk | Tokopedia", tokped_semua.height, "produk")
display(blibli_semua.group_by(
    nama_ada=pl.col("dari_nama").is_not_null(), variant_ada=pl.col("dari_variant").is_not_null()
).len().sort("len", descending=True))
display(tokped_semua.group_by(
    "ambigu", nama_ada=pl.col("dari_nama").is_not_null(), url_ada=pl.col("dari_url").is_not_null()
).len().sort("len", descending=True))

Blibli 2270 produk | Tokopedia 7787 produk


nama_ada,variant_ada,len
bool,bool,u32
true,false,1222
false,false,795
false,true,158
true,true,95


ambigu,nama_ada,url_ada,len
bool,bool,bool,u32
false,true,true,4655
false,false,false,2424
true,true,true,269
false,false,true,223
false,true,false,132
true,false,true,84


### Isi `variant` Blibli, produk yang beratnya hanya ada di cadangan, dan yang tidak sama dengan nama

In [44]:
with pl.Config(tbl_rows=-1):
    display(blibli_semua["variant"].value_counts(sort=True))

# Berat hanya ada di cadangan (nama tidak punya berat)
display(blibli_semua.filter(pl.col("dari_nama").is_null() & pl.col("dari_variant").is_not_null()))
display(tokped_semua.filter(pl.col("dari_nama").is_null() & pl.col("dari_url").is_not_null())
        .select("nama", "slug", "dari_url", "ambigu"))

# Nama dan cadangan sama-sama punya berat, tapi teksnya beda (cek mana yang benar)
display(blibli_semua.filter(pl.col("dari_nama").str.to_lowercase().str.replace_all(" ", "")
                            != pl.col("dari_variant").str.to_lowercase().str.replace_all(" ", "")))
display(tokped_semua.filter(~pl.col("ambigu") & (pl.col("dari_nama").str.to_lowercase().str.replace_all(" ", "")
                            != pl.col("dari_url").str.to_lowercase().str.replace_all(" ", "")))
        .select("nama", "slug", "dari_nama", "dari_url"))

variant,count
str,u32
null,1907
"""1/2 kg""",34
"""1 kg""",26
"""500 gram""",24
"""500gr""",19
"""500 g""",15
"""1 pcs""",11
"""500 gr""",11
"""250 gram""",10


id,nama,variant,dari_nama,dari_variant
str,str,str,str,str
"""DOD-70072-00911""","""Jeruk Medan""","""500 gram""",null,"""500 gram"""
"""SAX-70009-04986""","""Apel Dazzle Impor""","""500 g""",null,"""500 g"""
"""SAX-70009-02149""","""Anggur Hijau Shine Muscat Seedless Impor""","""250 g""",null,"""250 g"""
"""SMH-70020-00032""","""Stroberi Lokal Segar Ciwidey Strawberry Fresh Kualitas Tinggi""","""100g""",null,"""100g"""
"""NAS-70965-00593""","""Jeruk Limau/ Jeruk Limo/Jeruk Sambel""","""250 gr""",null,"""250 gr"""
"""BUS-70370-00105""","""KURMA AJWA AL MADINAH SUPER PREMIUM""","""500 GR""",null,"""500 GR"""
"""FRI-70030-00021""","""Buah Segar Anggur Autumn USA FRESHGO""","""1/4 kg""",null,"""1/4 kg"""
"""ZUB-70022-00035""","""Anggur Red Crimson Manis Seedless""","""1 kg""",null,"""1 kg"""
"""IBF-60022-00179""","""Buah Jeruk Impor Sunkist Lemon Santang Ponkan Wogan Murcot Mandarin :""","""LEMON 500 gram""",null,"""500 gram"""


nama,slug,dari_url,ambigu
str,str,str,bool
"""NANAS HONI SUNPRIDE MANIS SEGAR (1 PCS)""","""nanas honi sunpride manis segar 1 kg""","""1 kg""",false
"""BUAH ALPUKAT FRESH | ALPUKAT""","""buah alpukat fresh alpukat 450 500 gr 4c22f""","""500 gr""",true
"""KURMA AJWA KURMA AJWAH KURMA NABI ASLI DARI MADINAH""","""1 kg kurma ajwa kurma ajwah kurma nabi asli dari madinah 200 gram""","""1 kg""",false
"""JAMBU KRISTAL grade A BESAR isi (4 - 6 Buah) asli Tlogo , KEBUMEN""","""jambu kristal grade b fresh 1kg isi 5 7 buah asli pesisir kebumen buah segar dan…","""1kg""",false
"""LENGKENG IMPORT | BUAH LENGKENG FRESH (KHUSUS SAMEDAY DAN INSTAN)""","""lengkeng import buah lengkeng fresh khusus sameday dan instan 250 gr""","""250 gr""",false
"""Buah naga kuning EQUADOR super import Manis per 1 PCS""","""buah naga kuning equador super import manis per 1 pcs 300gr 179b9""","""300gr""",false
"""buah prunes plum""","""buah prunes plum 1kg""","""1kg""",false
"""Mangga Muda Apel Rasa Asam Manis Segar dan Bumbu Rujak bangkok By Savana Fruit""","""mangga muda rujak super berat 500 gr""","""500 gr""",false
"""Alpukat IMPERFECT Saleee""","""alpukat kelud imperfect berat 900 1kg""","""1kg""",true


id,nama,variant,dari_nama,dari_variant
str,str,str,str,str
"""IBF-60022-00178""","""Buah Pir Golden / Golden Sweet Pear IBAZA Buah Segar 1kg""","""1/2 kg""","""1kg""","""1/2 kg"""
"""IBF-60022-00365""","""Jeruk Mandarin Wokam per 1 kg Buah Segar Jeruk Wokam""","""500 gram""","""1 kg""","""500 gram"""
"""FRI-70030-00223""","""Melon Golden manis sunpride 1 bulat (1-1,5Kg) freshGO""","""1 kg""","""1,5Kg""","""1 kg"""
"""SA7-70003-00388""","""Lemon Lokal - 1kg""","""500gr""","""1kg""","""500gr"""
"""BAF-70007-00247""","""Buah Strawberry Strawberi Lokal 1 kg / Stroberi per pack""","""150 gram""","""1 kg""","""150 gram"""
"""IBF-60022-00180""","""Buah Pir Century / Pear Century - IBAZA Buah Segar 1kg""","""1/2 kg""","""1kg""","""1/2 kg"""
"""BAF-70007-00222""","""jeruk limo segar 250 gr / limau kasturi 500 gram / jeruk sambal sambel 1 Kg""","""250 gram""","""250 gr""","""250 gram"""
"""HAM-70080-00531""","""HasilSumberalaM Jeruk Aiyen Aiyuan Mandarin Orange Bibit Jepang IMPORT 900 gr - …","""1kg""","""900 gr""","""1kg"""
"""IBF-60022-00250""","""Buah Pir Packam/ Pear Packham - IBAZA Buah Segar 1kg""","""1/2 kg""","""1kg""","""1/2 kg"""


nama,slug,dari_nama,dari_url
str,str,str,str
"""KURMA TUNISIA TANGKAI 500 GRAM PREMIUM/KURMA MANIS ALAMI""","""tunisia tangkai 500g kurma tunis oasis kurma tangkai box 500g expt 2026""","""500 GRAM""","""500g"""
"""Kurma Bam Asli Iran 500gr Kurma Anggur Jumbo""","""kurma bam asli iran 500g madu iran kurma anggur""","""500gr""","""500g"""
"""Beet Root -/+ 900 - 1000 Gram / Pack Fresh Buah Bit Lokal Sayur Beetroot Organik…","""buah bit 1 kg beet root grade a organic fresh sayur ijo buah segar""","""1000 Gram""","""1 kg"""
"""Buah Kelengkeng Segar Fresh ± 450 gram""","""buah kelengkeng segar fresh 500 gram""","""450 gram""","""500 gram"""
"""Apel Fuji Cina 500 Gram Apple""","""apel fuji cina 500gr""","""500 Gram""","""500gr"""
"""Pepaya California 700 g - 800 g Per Buah""","""pepaya california medium 800 g 1 kg per buah""","""700 g""","""800 g"""
"""Kurma Medjool Palestine Ember 750gr Premium Dates""","""kurma medjool palestine premium 500 gram kemasan ember""","""750gr""","""500 gram"""
"""Pinang Muda Segar 500gr 1Kg | Buah Pinang Muda Fresh""","""250 gram buah pinang muda segar obat herbal buah jambe muda""","""500gr""","""250 gram"""
"""Jambu kristal purworejo 500g & 1kg""","""jambu kristal segar 1kg""","""500g""","""1kg"""


### Validasi: apakah berat cadangan sama dengan berat di nama?

Hanya produk yang **dua-duanya** punya berat. Keduanya diubah ke gram lalu dibandingkan.
Supaya perbandingannya adil, hanya angka bulat yang dihitung (`500 gr`, `1 kg`); pecahan, desimal, dan rentang (`1/2 kg`, `1,5 kg`, `1300/1500Gram`) dilewati dulu, karena aturannya belum dibuat.

In [45]:
def ke_gram(kolom):
    # "500 gr" -> 500, "1 Kg" -> 1000, "2 ons" -> 200. Selain angka bulat -> null.
    angka = pl.col(kolom).str.extract(r"^(\d+)\s*[a-zA-Z]", 1).cast(pl.Int64)
    satuan = pl.col(kolom).str.extract(r"([a-zA-Z]+)$", 1).str.to_lowercase()
    return (pl.when(satuan.str.starts_with("k")).then(angka * 1000)
            .when(satuan == "ons").then(angka * 100)
            .otherwise(angka))

banding_tokped = (
    tokped_semua.filter(~pl.col("ambigu"))
    .with_columns(gram_nama=ke_gram("dari_nama"), gram_url=ke_gram("dari_url"))
    .drop_nulls(["gram_nama", "gram_url"])
    .with_columns(sama=pl.col("gram_nama") == pl.col("gram_url"))
)
banding_blibli = (
    blibli_semua
    .with_columns(gram_nama=ke_gram("dari_nama"), gram_variant=ke_gram("dari_variant"))
    .drop_nulls(["gram_nama", "gram_variant"])
    .with_columns(sama=pl.col("gram_nama") == pl.col("gram_variant"))
)

print("Tokopedia: nama vs URL")
display(banding_tokped["sama"].value_counts())
display(banding_tokped.filter(~pl.col("sama")).select("nama", "slug", "gram_nama", "gram_url"))

print("Blibli: nama vs variant")
display(banding_blibli["sama"].value_counts())
display(banding_blibli.filter(~pl.col("sama")).select("id", "nama", "variant", "gram_nama", "gram_variant"))

Tokopedia: nama vs URL


sama,count
bool,u32
true,4562
false,89


nama,slug,gram_nama,gram_url
str,str,i64,i64
"""Buah Kelengkeng Segar Fresh ± 450 gram""","""buah kelengkeng segar fresh 500 gram""",450,500
"""Pepaya California 700 g - 800 g Per Buah""","""pepaya california medium 800 g 1 kg per buah""",700,800
"""Kurma Medjool Palestine Ember 750gr Premium Dates""","""kurma medjool palestine premium 500 gram kemasan ember""",750,500
"""Pinang Muda Segar 500gr 1Kg | Buah Pinang Muda Fresh""","""250 gram buah pinang muda segar obat herbal buah jambe muda""",500,250
"""Jambu kristal purworejo 500g & 1kg""","""jambu kristal segar 1kg""",500,1000
"""Mangga Muda Segar (±500 g/pack)""","""gendhis farm mangga muda segar 1 kg""",500,1000
"""Anggur Crimson 1kg 500 Scarlet Red Rose Seedless Manis""","""buah anggur seedless 500 gram anggur scarlet anggur crimson""",1000,500
"""buah alpukat endul surendul kemasan 1 kg dan 500g""","""coklat piramid mede almond berat 500g chocolate""",1000,500
"""Mangga Harum Manis 900 gr - 1000 gr""","""mangga harum manis 950g 1000g""",900,950


Blibli: nama vs variant


sama,count
bool,u32
false,20
true,55


id,nama,variant,gram_nama,gram_variant
str,str,str,i64,i64
"""IBF-60022-00365""","""Jeruk Mandarin Wokam per 1 kg Buah Segar Jeruk Wokam""","""500 gram""",1000,500
"""SA7-70003-00388""","""Lemon Lokal - 1kg""","""500gr""",1000,500
"""BAF-70007-00247""","""Buah Strawberry Strawberi Lokal 1 kg / Stroberi per pack""","""150 gram""",1000,150
"""HAM-70080-00531""","""HasilSumberalaM Jeruk Aiyen Aiyuan Mandarin Orange Bibit Jepang IMPORT 900 gr - …","""1kg""",900,1000
"""HAM-70080-01082""","""HasilSumberalaM LOLA Anggur Merah RED GLOBE Grapes IMPORT 1 kg""","""AUS 500gr""",1000,500
"""SA7-70003-00320""","""Lemon China - 1Kg""","""500gr""",1000,500
"""MAS-71112-00404""","""Stroberi Strawberry Segar Manis Jumbo Kemasan 500 Gram 100 Gram""","""100 gram""",500,100
"""GRS-28291-00461""","""Jeruk Nipis Asli Murah 500g 1kg""","""1 kg""",500,1000
"""MAS-71112-00180""","""Jeruk Limo Limau Sambel 250 Gram 100 Gram""","""100 gram""",250,100


### Satu nama produk Blibli di semua tanggal (ganti `cari_nama` lalu jalankan ulang)

Satu baris per (tanggal, `itemSku`). Kalau varian lain (misalnya 1 kg) ikut tertarik, `itemSku`-nya beda (`...-00002`) dan muncul sebagai baris sendiri.

In [46]:
cari_nama = "apel fuji super fresh"

baris = []
for berkas in sorted((akar / "blibli").glob("date=*/*.json")):
    for p in json.loads(berkas.read_text(encoding="utf-8"))["data"]["products"]:
        if cari_nama.lower() in p["name"].lower():
            baris.append({
                "tanggal": berkas.parent.name[5:],
                "id": p["id"],
                "item_sku": p.get("itemSku"),
                "nama": p["name"],
                "variant": p.get("variant"),
                "hasVariants": p.get("hasVariants"),
                "itemCount": p.get("itemCount"),
                "harga": p["price"]["salePrice"],
                "harga_coret": p["price"]["listPrice"],
                "penjual": p.get("merchantName"),
            })
hasil = pl.DataFrame(baris).sort("tanggal")
with pl.Config(tbl_rows=-1):
    display(hasil)
    display(hasil.group_by("id", "item_sku", "variant").agg(
        hari=pl.len(), harga=pl.col("harga").unique()))

tanggal,id,item_sku,nama,variant,hasVariants,itemCount,harga,harga_coret,penjual
str,str,str,str,str,bool,i64,i64,i64,str
"""2026-10-18""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-10-19""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-10-27""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-10-28""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-10-29""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-10-30""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-10-31""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-11-01""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""
"""2026-11-02""","""NAS-70965-00407""","""NAS-70965-00407-00001""","""Apel Fuji Super Fresh 1 Kg""","""500 gr""",true,2,46000,50000,"""Namora Sayur"""


id,item_sku,variant,hari,harga
str,str,str,u32,list[i64]
"""NAS-70965-00407""","""NAS-70965-00407-00001""","""500 gr""",14,[46000]


## 5. Halaman produk Tokopedia (Mangga Manalagi)

Kartu hasil cari Tokopedia tidak punya berat, stok, jumlah terjual pasti, dan info varian. Halaman produknya punya, dalam JSON `window.__cache` di HTML (tidak perlu Selenium).

- Produk = semua nama yang memuat **mangga** dan **manalagi** di tanggal Tokopedia terakhir.
- Setiap halaman disimpan ke `data_lake/halaman_tokopedia/date=.../<id>.json`. Yang sudah ada **tidak ditarik lagi**, jadi jalankan ulang cepat.
- Kadang Tokopedia mengirim halaman kosong (sekitar 22 KB, `__cache` kosong). Dicoba sampai 3 kali dengan jeda 6 detik.
- Pertama kali: sekitar 1–2 menit untuk ±22 produk.

In [47]:
import random
import time

from curl_cffi import requests

folder_cari = max((akar / "tokopedia").glob("date=*"))
folder_halaman = akar.parent / "halaman_tokopedia" / folder_cari.name
folder_halaman.mkdir(parents=True, exist_ok=True)

kartu = {}
for berkas in sorted(folder_cari.glob("*.json")):
    for p in json.loads(berkas.read_text(encoding="utf-8"))[0]["data"]["searchProductV5"]["data"]["products"]:
        nama = p["name"].lower()
        if "mangga" in nama and "manalagi" in nama:
            kartu[p["id"]] = p
print(folder_cari.name, "-", len(kartu), "produk mangga manalagi")


def ambil_cache(sesi, url):
    """HTML halaman produk -> dict window.__cache (kosong kalau Tokopedia mengirim halaman kosong)."""
    teks = sesi.get(url, timeout=60).text
    if "window.__cache=" not in teks:
        return {}
    awal = teks.index("window.__cache=") + len("window.__cache=")
    cache, _ = json.JSONDecoder().raw_decode(teks[awal:])
    return cache


sesi = None
for id_, p in kartu.items():
    jalur = folder_halaman / f"{id_}.json"
    if jalur.exists():
        continue
    if sesi is None:
        sesi = requests.Session(impersonate="chrome")
        sesi.get("https://www.tokopedia.com/", timeout=60)
    cache = {}
    for coba in range(1, 4):
        cache = ambil_cache(sesi, p["url"].split("?")[0])
        if cache:
            break
        time.sleep(6)
    if cache:
        jalur.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
        print("ok   ", id_, f"(coba {coba})")
    else:
        print("GAGAL", id_, p["name"][:50])
    time.sleep(random.uniform(1.5, 3.0))
print("tersimpan:", len(list(folder_halaman.glob("*.json"))), "halaman di", folder_halaman)

date=2026-11-07 - 22 produk mangga manalagi
ok    13619735933 (coba 1)
ok    4091182300 (coba 2)
ok    8061245026 (coba 1)
ok    1256380209 (coba 2)
tersimpan: 22 halaman di data_lake\halaman_tokopedia\date=2026-11-07


### Kartu cari vs halaman produk

`_kartu` = dari hasil cari (data harian kita), `_hal` = dari halaman produk. `varian_dari` terisi kalau kartu ini ternyata **satu opsi** dari produk induk (`parentID`).

In [48]:
def jenis(cache, nama_tipe):
    """Ambil objek pertama di cache dengan __typename tertentu."""
    return next((v for v in cache.values() if isinstance(v, dict) and v.get("__typename") == nama_tipe), {})


baris = []
for id_, p in kartu.items():
    jalur = folder_halaman / f"{id_}.json"
    label = {g["position"]: g["title"] for g in p["labelGroups"]}
    b = {
        "id": str(id_),
        "nama": p["name"],
        "toko": p["shop"]["name"],
        "harga_kartu": p["price"]["number"],
        "coret_kartu": p["price"]["original"] or None,
        "label_promo_kartu": label.get("promotion_label"),
        "terjual_kartu": label.get("ri_product_credibility"),
        "rating_kartu": p["rating"],
    }
    if jalur.exists():
        cache = json.loads(jalur.read_text(encoding="utf-8"))
        info = jenis(cache, "pdpBasicInfo")
        harga = jenis(cache, "pdpContentSnapshotPrice")
        varian = jenis(cache, "pdpContentSnapshotVariant")
        b |= {
            "harga_hal": harga.get("value"),
            "coret_hal": harga.get("slashPriceFmt") or None,
            "berat_hal": f"{info.get('weight')} {info.get('weightUnit', '').lower()}",
            "stok_hal": jenis(cache, "pdpContentSnapshotStock").get("value"),
            "terjual_hal": jenis(cache, "pdpTxStats").get("countSold"),
            "rating_hal": jenis(cache, "pdpStats").get("rating"),
            "jumlah_rating_hal": jenis(cache, "pdpStats").get("countReview"),
            "min_beli_hal": info.get("minOrder"),
            "kampanye_aktif_hal": jenis(cache, "pdpContentSnapshotCampaign").get("isActive"),
            "varian_dari": varian.get("parentID") if varian.get("isVariant") else None,
            "dibuat_hal": (info.get("createdAt") or "")[:10],
        }
    baris.append(b)

halaman = pl.DataFrame(baris, infer_schema_length=None)
with pl.Config(tbl_rows=-1, fmt_str_lengths=45):
    display(halaman)

ada = halaman.filter(pl.col("harga_hal").is_not_null())
print("halaman tersimpan:", ada.height, "/", halaman.height, "(jalankan ulang sel tarik untuk yang belum)")
print("harga kartu == harga halaman:", (ada["harga_kartu"] == ada["harga_hal"].cast(pl.Int64)).sum(), "/", ada.height)
print("kartu ada harga coret, halaman tidak:", ada.filter(pl.col("coret_kartu").is_not_null() & pl.col("coret_hal").is_null()).height)
print("kartu = salah satu opsi produk induk:", ada["varian_dari"].is_not_null().sum())

id,nama,toko,harga_kartu,coret_kartu,label_promo_kartu,terjual_kartu,rating_kartu,harga_hal,coret_hal,berat_hal,stok_hal,terjual_hal,rating_hal,jumlah_rating_hal,min_beli_hal,kampanye_aktif_hal,varian_dari,dibuat_hal
str,str,str,i64,str,str,str,str,i64,null,str,str,str,f64,str,i64,bool,str,str
"""353179938""","""Mangga Manalagi Super Kiloan Raja Alpukat""","""RAJA ALPUKAT""",29000,null,null,"""1rb+ terjual""","""4.7""",29000,null,"""1 kilogram""","""509""","""1343""",4.7,"""289""",1,false,null,"""2024-09-01"""
"""102383650601""","""Mangga Manalagi Segar 1Kg – Manis dan Daging …","""Mracang Market""",14000,null,null,"""12 terjual""","""5.0""",14000,null,"""1 kilogram""","""56""","""12""",5.0,"""1""",1,false,null,"""2025-10-08"""
"""1178532826""","""Mangga Manalagi /1 kg""","""Pasar induk kramatjati online""",31900,null,null,"""100+ terjual""","""5.0""",31900,null,"""1 kilogram""","""47""","""126""",5.0,"""48""",1,false,null,"""2024-09-03"""
"""15003295432""","""Buah mangga manalagi manis 1 kg jual buah bua…","""YukBeli Toserba""",49999,null,null,"""17 terjual""","""5.0""",49999,null,"""1 kilogram""","""86""","""17""",5.0,"""4""",1,false,null,"""2024-09-01"""
"""14020755371""","""Mangga Manalagi Potong Manis""","""Siti Fresh""",10000,null,null,"""26 terjual""","""5.0""",10000,null,"""0.35 kilogram""","""982""","""26""",5.0,"""12""",1,false,null,"""2024-09-15"""
"""6854272732""","""mangga manalagi 1 kg""","""hafis-online""",55000,null,null,"""29 terjual""","""5.0""",55000,null,"""1 kilogram""","""996""","""29""",5.0,"""9""",2,false,null,"""2024-09-13"""
"""102472942809""","""MANGGA MANALAGI MANIS, (1KG) flash sale Buah""","""MAS JOKO BUAH SEGAR""",45500,null,null,"""70+ terjual""","""5.0""",45500,null,"""1 kilogram""","""165""","""76""",5.0,"""12""",1,false,null,"""2025-10-25"""
"""13619735933""","""mangga manalagi fresh 1 kg / mangga madu mana…","""rumindonesia""",32500,null,null,"""100+ terjual""","""5.0""",32500,null,"""1 kilogram""","""60""","""141""",5.0,"""27""",1,false,null,"""2024-09-16"""
"""4091182300""","""Mangga Manalagi manis segar 1 kg""","""PartnerBuah""",55000,null,null,"""12 terjual""","""5.0""",55000,null,"""1 kilogram""","""88""","""12""",5.0,"""3""",1,false,null,"""2024-09-04"""


halaman tersimpan: 22 / 22 (jalankan ulang sel tarik untuk yang belum)
harga kartu == harga halaman: 22 / 22
kartu ada harga coret, halaman tidak: 2
kartu = salah satu opsi produk induk: 7
